### This notebook script read pre-saved pressure data from folder and generate explosion sound.
### 고퀄리티의 음성을 만드려면 1마이크 세컨 미만의 dt로 시뮬레이션을 돌려야하는데 이는 불가능하 굳이 의미 모르겠어 포기함.

In [1]:
import torch
import numpy as np
import os
from tqdm import tqdm
import gc
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.io import wavfile
import librosa



from CFD import RadiativeTransfer
from CFD import AdvectingField
from CFD import create_sphere_mask


In [2]:
# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
    
#this is where simulation results were saved.
folder_path = "/mnt/storage/pressure"
params = torch.load("/mnt/storage/params.pt")

Using device: cuda


In [3]:
boundary_band_radius = params["boundary_band_radius"] + 0.1

d = boundary_band_radius / 3 ** 0.5
center_pos = (0.5, 0.5, 0.5)
ear_distance = 0.02

direction = np.array([0, 2**0.5/4, -2**0.5/4]) * ear_distance
left_ear_pos  = (center_pos + direction)
right_ear_pos = (center_pos - direction)

DS = (params["DZ"], params["DY"], params["DX"])
left_ear_index = tuple((left_ear_pos / DS).astype(int))
right_ear_index = tuple((right_ear_pos / DS).astype(int))

print(left_ear_index)
print(right_ear_index)

'''
#Get file names ready
file_names = sorted([
    f for f in os.listdir(folder_path)
    if os.path.isfile(os.path.join(folder_path, f)) and f.endswith(".pt")
])

left_ear_pressure = []
right_ear_pressure = []
times = []

for file_name in tqdm(file_names):
    file_path = os.path.join(folder_path, file_name)
    pressure_field = torch.load(file_path, map_location='cpu', mmap=True, weights_only=True)
    time = int(file_name[:-3])
    
    # 필요한 값만 뽑아서 리스트에 저장 (텐서라면 .item()이나 .cpu()로 가볍게)
    left_ear_pressure.append(pressure_field[left_ear_index].item())
    right_ear_pressure.append(pressure_field[right_ear_index].item())
    times.append(time)
    
    # 메모리 강제 해제
    del pressure_field
    gc.collect()
torch.save(left_ear_pressure, "./sound_generation/left_ear_pressure.pt")
torch.save(right_ear_pressure, "./sound_generation/right_ear_pressure.pt")
torch.save(times, "./sound_generation/times.pt")
'''

(np.int64(50), np.int64(304), np.int64(295))
(np.int64(50), np.int64(295), np.int64(304))


'\n#Get file names ready\nfile_names = sorted([\n    f for f in os.listdir(folder_path)\n    if os.path.isfile(os.path.join(folder_path, f)) and f.endswith(".pt")\n])\n\nleft_ear_pressure = []\nright_ear_pressure = []\ntimes = []\n\nfor file_name in tqdm(file_names):\n    file_path = os.path.join(folder_path, file_name)\n    pressure_field = torch.load(file_path, map_location=\'cpu\', mmap=True, weights_only=True)\n    time = int(file_name[:-3])\n\n    # 필요한 값만 뽑아서 리스트에 저장 (텐서라면 .item()이나 .cpu()로 가볍게)\n    left_ear_pressure.append(pressure_field[left_ear_index].item())\n    right_ear_pressure.append(pressure_field[right_ear_index].item())\n    times.append(time)\n\n    # 메모리 강제 해제\n    del pressure_field\n    gc.collect()\ntorch.save(left_ear_pressure, "./sound_generation/left_ear_pressure.pt")\ntorch.save(right_ear_pressure, "./sound_generation/right_ear_pressure.pt")\ntorch.save(times, "./sound_generation/times.pt")\n'

In [8]:
# 1. 파라미터 설정
pb_speed = 0.1  # 0.006
p0 = params["p0"]              # 대기압 상수
fs_target = 44100              # 최종 출력 샘플링 레이트

# 2. 데이터 로드 및 초기 처리 (물리적 시간 축 유지)
left_ear_raw = torch.load("./sound_generation/left_ear_pressure.pt")
right_ear_raw = torch.load("./sound_generation/right_ear_pressure.pt")
# 물리적 시간(micro-sec)을 초(sec) 단위로 변환 (누적합 필요시 적용)
times_raw = torch.load("./sound_generation/times.pt") 
t_seconds = torch.tensor(times_raw).numpy() * 1e-6 

# 대기압 제거 (변동 압력 추출)
p_left = (torch.tensor(left_ear_raw) * p0).numpy() - 101325.0
p_right = (torch.tensor(right_ear_raw) * p0).numpy() - 101325.0

# 3. 고해상도 중간 데이터 생성 (Source Sampling)
# librosa 처리를 위해 먼저 일정한 간격의 높은 샘플링 레이트로 보간합니다.
# 원본 dt가 20us이므로 fs_orig를 50,000Hz 정도로 잡습니다.
fs_orig = int(1 / (np.mean(np.diff(t_seconds)) + 1e-9)) 
t_uniform_orig = np.linspace(t_seconds[0], t_seconds[-1], len(t_seconds))

f_l = interp1d(t_seconds, p_left, kind='cubic', fill_value="extrapolate")
f_r = interp1d(t_seconds, p_right, kind='cubic', fill_value="extrapolate")

p_l_orig = f_l(t_uniform_orig)
p_r_orig = f_r(t_uniform_orig)

# 4. Librosa Time-Stretch 적용 (음정 유지, 시간만 연장)
# rate < 1.0 이면 느려집니다. 0.006은 매우 작으므로 알고리즘적 한계가 올 수 있습니다.
# 만약 소리가 끊기면 pb_speed를 조금 높여보세요 (예: 0.05)
p_l_stretched = librosa.effects.time_stretch(p_l_orig, rate=pb_speed)
p_r_stretched = librosa.effects.time_stretch(p_r_orig, rate=pb_speed)

# 5. 최종 리샘플링 (Target 44100Hz에 맞춤)
# 스트레칭된 데이터의 길이에 맞춰 최종 결과물을 만듭니다.
p_l_final = librosa.resample(p_l_stretched, orig_sr=fs_orig, target_sr=fs_target)
p_r_final = librosa.resample(p_r_stretched, orig_sr=fs_orig, target_sr=fs_target)

# 6. 정규화 및 저장
max_val = max(np.max(np.abs(p_l_final)), np.max(np.abs(p_r_final)))
if max_val > 0:
    p_l_norm = p_l_final / max_val
    p_r_norm = p_r_final / max_val

stereo_audio = np.vstack((p_l_norm, p_r_norm)).T
audio_int16 = (stereo_audio * 32767).astype(np.int16)

wavfile.write("explosion_high_quality_stretch.wav", fs_target, audio_int16)

print(f"오디오 생성 완료: {len(p_l_final)/fs_target:.2f} 초")

오디오 생성 완료: 0.99 초


In [5]:
# 1. 그래프 크기와 해상도 설정 (고퀄리티용)
plt.figure(figsize=(10, 5), dpi=100)

# 2. 데이터 플롯 (색상: 파란색, 선 굵기: 1.5)
plt.plot(t_uniform, p_left_norm, color='b', linewidth=1.5, label='Pressure at left ear')
plt.plot(t_uniform, p_right_norm, color='r', linewidth=1.5, label='Pressure at right ear')

# 3. 축 레이블 및 타이틀 설정
plt.xlabel('Time ($\mu s$)') # 마이크로 세컨즈 단위 표기
plt.ylabel('Pressure (Pa)')
plt.title('Explosion Pressure Waveform')

# 4. 그리드 및 범례 추가
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()

# 5. 화면 표시
plt.show()

<>:9: SyntaxWarning: invalid escape sequence '\m'
<>:9: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_40090/1401786786.py:9: SyntaxWarning: invalid escape sequence '\m'
  plt.xlabel('Time ($\mu s$)') # 마이크로 세컨즈 단위 표기
/tmp/ipykernel_40090/1401786786.py:9: SyntaxWarning: invalid escape sequence '\m'
  plt.xlabel('Time ($\mu s$)') # 마이크로 세컨즈 단위 표기


NameError: name 't_uniform' is not defined

<Figure size 1000x500 with 0 Axes>